In [8]:
# pip install pulp
import pulp
import math
import random

# -----------------------------
# PARAMETERS
# -----------------------------
N = 13
customers = list(range(1, N+1))
depot = 0
nodes = [depot] + customers

m = 1
vehicles = [3]

speed = 25.23     # mph
H = 240           # shift minutes
BIG = 10000

# -----------------------------
# Package demand
# -----------------------------
q = {j:1 for j in customers}
Q = {1:999}

# -----------------------------
# MOST service times
# -----------------------------
s = {}

for j in customers:
    service_type = random.choice(["house","company","apartment"])
    
    if service_type == "house":
        s[j] = 12.7
    elif service_type == "company":
        s[j] = 13.4
    else:
        s[j] = 24.6

# -----------------------------
# Coordinates
# customers located 5–7 miles from depot
# -----------------------------
coords = {0:(0,0)}

for j in customers:
    angle = 2*math.pi*j/N
    radius = random.uniform(5,7)
    
    coords[j] = (
        radius*math.cos(angle),
        radius*math.sin(angle)
    )

# -----------------------------
# Distance matrix
# -----------------------------
d = {}

for i in nodes:
    for j in nodes:
        if i != j:
            x1,y1 = coords[i]
            x2,y2 = coords[j]
            
            d[i,j] = math.sqrt(
                (x1-x2)**2 + (y1-y2)**2
            )

# -----------------------------
# MODEL
# -----------------------------
model = pulp.LpProblem(
    "MILP_VRP_13_Customers",
    pulp.LpMaximize
)

# route variable
x = pulp.LpVariable.dicts(
    "x",
    [(i,j) for i in nodes for j in nodes if i!=j],
    cat="Binary"
)

# customer served variable
y = pulp.LpVariable.dicts(
    "y",
    customers,
    cat="Binary"
)

# MTZ variables
u = pulp.LpVariable.dicts(
    "u",
    customers,
    lowBound=1,
    upBound=N,
    cat="Continuous"
)

# -----------------------------
# OBJECTIVE
# maximize deliveries first
# minimize distance second
# -----------------------------
model += (
    BIG * pulp.lpSum(y[j] for j in customers)
    -
    pulp.lpSum(
        d[i,j]*x[i,j]
        for i in nodes
        for j in nodes
        if i!=j
    )
)

# -----------------------------
# CUSTOMER VISIT CONSTRAINT
# optional delivery
# -----------------------------
for j in customers:
    
    model += (
        pulp.lpSum(
            x[i,j]
            for i in nodes if i!=j
        ) == y[j]
    )
    
    model += (
        pulp.lpSum(
            x[j,k]
            for k in nodes if j!=k
        ) == y[j]
    )

# -----------------------------
# DEPOT START
# -----------------------------
model += (
    pulp.lpSum(
        x[0,j] for j in customers
    ) <= 1
)

# -----------------------------
# DEPOT RETURN
# -----------------------------
model += (
    pulp.lpSum(
        x[i,0] for i in customers
    ) <= 1
)

# -----------------------------
# FLOW CONSERVATION
# -----------------------------
for h in customers:
    
    model += (
        pulp.lpSum(
            x[i,h]
            for i in nodes if i!=h
        )
        ==
        pulp.lpSum(
            x[h,j]
            for j in nodes if j!=h
        )
    )

# -----------------------------
# CAPACITY CONSTRAINT
# -----------------------------
model += (
    pulp.lpSum(
        q[j]*y[j]
        for j in customers
    )
    <= Q[1]
)

# -----------------------------
# TRAVEL TIME
# -----------------------------
travel_time = pulp.lpSum(
    (d[i,j]/speed)*60*x[i,j]
    for i in nodes
    for j in nodes
    if i!=j
)

# -----------------------------
# SERVICE TIME
# -----------------------------
service_time = pulp.lpSum(
    s[j]*y[j]
    for j in customers
)

# -----------------------------
# SHIFT LIMIT
# -----------------------------
model += (
    travel_time + service_time <= H
)

# -----------------------------
# MTZ SUBTOUR ELIMINATION
# -----------------------------
for i in customers:
    for j in customers:
        if i != j:
            model += (
                u[i] - u[j] + N*x[i,j]
                <= N-1
            )

# -----------------------------
# SOLVE
# -----------------------------
solver = pulp.PULP_CBC_CMD(msg=True)
model.solve(solver)

print("Status:", pulp.LpStatus[model.status])

if pulp.LpStatus[model.status] not in ["Optimal","Feasible"]:
    print("No feasible route found")
    
else:
    
    served = [
        j for j in customers
        if pulp.value(y[j]) > 0.5
    ]
    
    missed = N - len(served)
    
    print("Completed Deliveries:", len(served))
    print("Missed Deliveries:", missed)
    print("Penalty Cost: $", round(missed*1.20,2))
    
    # -------------------------
    # Route reconstruction
    # -------------------------
    route = [0]
    current = 0
    
    while True:
        
        next_node = None
        
        for j in nodes:
            if current != j and (current,j) in x:
                
                if pulp.value(x[current,j]) > 0.5:
                    next_node = j
                    break
        
        if next_node is None:
            break
        
        route.append(next_node)
        
        if next_node == 0:
            break
        
        current = next_node
    
    print("Optimal Route:", route)
    
    # -------------------------
    # Distance
    # -------------------------
    total_distance = 0
    
    for i in range(len(route)-1):
        total_distance += d[
            route[i],
            route[i+1]
        ]
    
    print("Total Distance:",
          round(total_distance,2),
          "miles")
    
    # -------------------------
    # Travel time
    # -------------------------
    total_travel_time = (
        total_distance/speed
    )*60
    
    print("Travel Time:",
          round(total_travel_time,2),
          "min")
    
    # -------------------------
    # Service time
    # -------------------------
    total_service_time = sum(
        s[j] for j in served
    )
    
    print("Service Time:",
          round(total_service_time,2),
          "min")
    
    # -------------------------
    # Total elapsed time
    # -------------------------
    total_elapsed = (
        total_travel_time
        +
        total_service_time
    )
    
    print("Elapsed Time:",
          round(total_elapsed,2),
          "min")
    
    # -------------------------
    # Utilization
    # -------------------------
    utilization = (
        total_elapsed/H
    )*100
    
    print("Utilization:",
          round(utilization,2),
          "%")

Status: Optimal
Completed Deliveries: 9
Missed Deliveries: 4
Penalty Cost: $ 4.8
Optimal Route: [0, 4, 5, 6, 7, 8, 9, 10, 11, 12, 0]
Total Distance: 34.51 miles
Travel Time: 82.08 min
Service Time: 151.4 min
Elapsed Time: 233.48 min
Utilization: 97.28 %


Allowed arcs: 240
Original full arcs: 900

SOLVER RESULT
Status: Optimal
Runtime seconds: 264.38
Runtime minutes: 4.41
Runtime hours: 0.0734

DELIVERY SUMMARY
Completed deliveries: 30
Missed deliveries: 0
Penalty cost: $ 0.0

VEHICLE 1
Route: [0, 11, 10, 9, 8, 7, 6, 5, 4, 3, 0]
Served customers: 9
Distance: 22.74 miles
Travel time: 54.08 min
Service time: 150.7 min
Elapsed time: 204.78 min
Utilization: 85.32 %

VEHICLE 2
Route: [0, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 0]
Served customers: 10
Distance: 24.3 miles
Travel time: 57.78 min
Service time: 163.4 min
Elapsed time: 221.18 min
Utilization: 92.16 %

VEHICLE 3
Route: [0, 26, 27, 28, 29, 30, 1, 2, 0]
Served customers: 7
Distance: 19.81 miles
Travel time: 47.11 min
Service time: 148.4 min
Elapsed time: 195.51 min
Utilization: 81.46 %

VEHICLE 4
Route: [0, 15, 14, 13, 12, 0]
Served customers: 4
Distance: 14.87 miles
Travel time: 35.35 min
Service time: 86.5 min
Elapsed time: 121.85 min
Utilization: 50.77 %

Vehicle 5: Not used

FLE

In [12]:
# pip install pulp

import pulp
import math
import random
import time
import pandas as pd

# =====================================================
# GLOBAL PARAMETERS
# =====================================================
N = 30
customers = list(range(1, N + 1))
depot = 0
nodes = [depot] + customers

speed = 25.23
H = 240
BIG = 10000
PENALTY_PER_MISSED = 1.20
NEAREST_NEIGHBORS = 8
SOLVER_TIME_LIMIT = 600  # 10 minutes per vehicle scenario

random.seed(42)

# =====================================================
# DEMAND
# =====================================================
q = {j: 1 for j in customers}

# =====================================================
# SERVICE TIMES
# =====================================================
s = {}
customer_type = {}

for j in customers:
    service_type = random.choice(["house", "company", "apartment"])
    customer_type[j] = service_type

    if service_type == "house":
        s[j] = 12.7
    elif service_type == "company":
        s[j] = 13.4
    else:
        s[j] = 24.6

# =====================================================
# COORDINATES
# =====================================================
coords = {0: (0, 0)}

for j in customers:
    angle = 2 * math.pi * j / N
    radius = random.uniform(5, 7)

    coords[j] = (
        radius * math.cos(angle),
        radius * math.sin(angle)
    )

# =====================================================
# DISTANCE MATRIX
# =====================================================
d = {}

for i in nodes:
    for j in nodes:
        if i != j:
            x1, y1 = coords[i]
            x2, y2 = coords[j]

            d[i, j] = math.sqrt(
                (x1 - x2) ** 2 +
                (y1 - y2) ** 2
            )

# =====================================================
# ALLOWED ARCS
# =====================================================
allowed_arcs = set()

for i in nodes:

    if i == depot:
        neighbors = customers

    else:
        nearest = sorted(
            [(j, d[i, j]) for j in customers if j != i],
            key=lambda x: x[1]
        )[:NEAREST_NEIGHBORS]

        neighbors = [j for j, _ in nearest]
        neighbors.append(depot)

    for j in neighbors:
        if i != j:
            allowed_arcs.add((i, j))

allowed_arcs = list(allowed_arcs)

print("Allowed arcs:", len(allowed_arcs))

# =====================================================
# MILP FUNCTION
# =====================================================
def solve_vrp_for_vehicle_count(m):

    vehicles = list(range(1, m + 1))
    Q = {k: 999 for k in vehicles}

    model = pulp.LpProblem(
        f"MILP_VRP_{m}_Vehicles",
        pulp.LpMaximize
    )

    x = pulp.LpVariable.dicts(
        "x",
        [(i, j, k) for (i, j) in allowed_arcs for k in vehicles],
        cat="Binary"
    )

    y = pulp.LpVariable.dicts(
        "y",
        customers,
        cat="Binary"
    )

    used = pulp.LpVariable.dicts(
        "used",
        vehicles,
        cat="Binary"
    )

    u = pulp.LpVariable.dicts(
        "u",
        [(j, k) for j in customers for k in vehicles],
        lowBound=1,
        upBound=N,
        cat="Continuous"
    )

    # Objective
    model += (
        BIG * pulp.lpSum(y[j] for j in customers)
        -
        pulp.lpSum(
            d[i, j] * x[i, j, k]
            for (i, j) in allowed_arcs
            for k in vehicles
        )
    )

    # Customer visit
    for j in customers:

        model += (
            pulp.lpSum(
                x[i, j, k]
                for (i, jj) in allowed_arcs
                for k in vehicles
                if jj == j
            ) == y[j]
        )

        model += (
            pulp.lpSum(
                x[j, i, k]
                for (jj, i) in allowed_arcs
                for k in vehicles
                if jj == j
            ) == y[j]
        )

    # Depot start and return
    for k in vehicles:

        model += (
            pulp.lpSum(
                x[depot, j, k]
                for (i, j) in allowed_arcs
                if i == depot
            ) == used[k]
        )

        model += (
            pulp.lpSum(
                x[i, depot, k]
                for (i, j) in allowed_arcs
                if j == depot
            ) == used[k]
        )

    # Flow conservation
    for k in vehicles:
        for h in customers:

            model += (
                pulp.lpSum(
                    x[i, h, k]
                    for (i, j) in allowed_arcs
                    if j == h
                )
                ==
                pulp.lpSum(
                    x[h, j, k]
                    for (i, j) in allowed_arcs
                    if i == h
                )
            )

    # Capacity
    for k in vehicles:

        model += (
            pulp.lpSum(
                q[j] *
                pulp.lpSum(
                    x[i, j, k]
                    for (i, jj) in allowed_arcs
                    if jj == j
                )
                for j in customers
            )
            <= Q[k] * used[k]
        )

    # Shift constraint
    for k in vehicles:

        travel_time = pulp.lpSum(
            (d[i, j] / speed) * 60 * x[i, j, k]
            for (i, j) in allowed_arcs
        )

        service_time = pulp.lpSum(
            s[j] *
            pulp.lpSum(
                x[i, j, k]
                for (i, jj) in allowed_arcs
                if jj == j
            )
            for j in customers
        )

        model += travel_time + service_time <= H * used[k]

    # MTZ subtour elimination
    for k in vehicles:
        for i in customers:
            for j in customers:
                if i != j and (i, j) in allowed_arcs:
                    model += (
                        u[i, k] - u[j, k] + N * x[i, j, k]
                        <= N - 1
                    )

    # Symmetry breaking
    for k in range(1, m):
        model += used[k] >= used[k + 1]

    # Solve
    start_time = time.time()

    solver = pulp.PULP_CBC_CMD(
        msg=False,
        timeLimit=SOLVER_TIME_LIMIT
    )

    model.solve(solver)

    runtime = time.time() - start_time
    status = pulp.LpStatus[model.status]

    served = [
        j for j in customers
        if pulp.value(y[j]) is not None and pulp.value(y[j]) > 0.5
    ]

    missed = N - len(served)
    penalty = missed * PENALTY_PER_MISSED

    vehicle_outputs = []

    total_distance = 0
    total_elapsed = 0

    for k in vehicles:

        if pulp.value(used[k]) is None or pulp.value(used[k]) < 0.5:

            vehicle_outputs.append({
                "Vehicle Scenario": m,
                "Vehicle": k,
                "Used": "No",
                "Route": [],
                "Served": 0,
                "Distance": 0,
                "Travel Time": 0,
                "Service Time": 0,
                "Elapsed Time": 0,
                "Utilization %": 0
            })

            continue

        route = [depot]
        current = depot
        visited_check = set()

        while True:

            next_node = None

            for (i, j) in allowed_arcs:
                if i == current:
                    val = pulp.value(x[i, j, k])
                    if val is not None and val > 0.5:
                        next_node = j
                        break

            if next_node is None:
                break

            route.append(next_node)

            if next_node == depot:
                break

            if next_node in visited_check:
                break

            visited_check.add(next_node)
            current = next_node

        route_customers = [r for r in route if r != depot]

        route_distance = sum(
            d[a, b]
            for a, b in zip(route[:-1], route[1:])
        )

        route_travel_time = route_distance / speed * 60
        route_service_time = sum(s[j] for j in route_customers)
        route_elapsed_time = route_travel_time + route_service_time
        utilization = route_elapsed_time / H * 100

        total_distance += route_distance
        total_elapsed += route_elapsed_time

        vehicle_outputs.append({
            "Vehicle Scenario": m,
            "Vehicle": k,
            "Used": "Yes",
            "Route": route,
            "Served": len(route_customers),
            "Distance": round(route_distance, 2),
            "Travel Time": round(route_travel_time, 2),
            "Service Time": round(route_service_time, 2),
            "Elapsed Time": round(route_elapsed_time, 2),
            "Utilization %": round(utilization, 2)
        })

    summary = {
        "Vehicles": m,
        "Status": status,
        "Completed Deliveries": len(served),
        "Missed Deliveries": missed,
        "Penalty Cost": round(penalty, 2),
        "Total Fleet Distance": round(total_distance, 2),
        "Total Fleet Elapsed Time": round(total_elapsed, 2),
        "Runtime Seconds": round(runtime, 2),
        "Runtime Minutes": round(runtime / 60, 2)
    }

    return summary, vehicle_outputs


# =====================================================
# RUN MULTIPLE VEHICLE SCENARIOS
# =====================================================
vehicle_scenarios = [2, 3, 4, 5]

summary_results = []
vehicle_level_results = []

for m in vehicle_scenarios:

    print(f"\nSolving MILP for {m} vehicles...")

    summary, vehicle_outputs = solve_vrp_for_vehicle_count(m)

    summary_results.append(summary)
    vehicle_level_results.extend(vehicle_outputs)

# =====================================================
# DISPLAY SUMMARY TABLE
# =====================================================
summary_df = pd.DataFrame(summary_results)

print("\n==============================")
print("SUMMARY COMPARISON")
print("==============================")
print(summary_df)

# =====================================================
# DISPLAY VEHICLE-LEVEL TABLE
# =====================================================
vehicle_df = pd.DataFrame(vehicle_level_results)

print("\n==============================")
print("EACH VEHICLE ANALYSIS")
print("==============================")
print(vehicle_df)

# Optional: save to Excel
summary_df.to_excel("MILP_vehicle_summary.xlsx", index=False)
vehicle_df.to_excel("MILP_each_vehicle_analysis.xlsx", index=False)

print("\nExcel files saved:")
print("MILP_vehicle_summary.xlsx")
print("MILP_each_vehicle_analysis.xlsx")

Allowed arcs: 300

Solving MILP for 2 vehicles...

Solving MILP for 3 vehicles...

Solving MILP for 4 vehicles...

Solving MILP for 5 vehicles...

SUMMARY COMPARISON
   Vehicles   Status  Completed Deliveries  Missed Deliveries  Penalty Cost  \
0         2  Optimal                    21                  9          10.8   
1         3  Optimal                    29                  1           1.2   
2         4  Optimal                    30                  0           0.0   
3         5  Optimal                    30                  0           0.0   

   Total Fleet Distance  Total Fleet Elapsed Time  Runtime Seconds  \
0                 49.51                    457.23           600.92   
1                 70.94                    693.11           605.99   
2                 81.71                    743.32           240.49   
3                 81.85                    743.65           601.14   

   Runtime Minutes  
0            10.02  
1            10.10  
2             4.01  
3  

In [14]:
# pip install pulp pandas openpyxl

import pulp
import math
import random
import time
import pandas as pd

# =====================================================
# GLOBAL PARAMETERS
# =====================================================
N = 30
customers = list(range(1, N + 1))
depot = 0
nodes = [depot] + customers

speed = 25.23              # mph
H = 240                    # 4-hour shift in minutes
BIG = 10000
PENALTY_PER_MISSED = 1.20

SOLVER_TIME_LIMIT = 600    # seconds per scenario; change to 1800 for 30 min
random.seed(42)

# =====================================================
# PACKAGE DEMAND AND SERVICE TIMES
# =====================================================
q = {j: 1 for j in customers}

s = {}
customer_type = {}

for j in customers:
    service_type = random.choice(["house", "company", "apartment"])
    customer_type[j] = service_type

    if service_type == "house":
        s[j] = 12.7
    elif service_type == "company":
        s[j] = 13.4
    else:
        s[j] = 24.6

# =====================================================
# CUSTOMER COORDINATES: 5–7 MILES FROM DEPOT
# =====================================================
coords = {0: (0, 0)}

for j in customers:
    angle = 2 * math.pi * j / N
    radius = random.uniform(5, 7)

    coords[j] = (
        radius * math.cos(angle),
        radius * math.sin(angle)
    )

# =====================================================
# DISTANCE MATRIX
# =====================================================
d = {}

for i in nodes:
    for j in nodes:
        if i != j:
            x1, y1 = coords[i]
            x2, y2 = coords[j]

            d[i, j] = math.sqrt(
                (x1 - x2) ** 2 +
                (y1 - y2) ** 2
            )

# =====================================================
# MILP SOLVER FUNCTION
# =====================================================
def solve_milp_vrp(m):

    vehicles = list(range(1, m + 1))
    Q = {k: 999 for k in vehicles}

    model = pulp.LpProblem(
        f"MILP_VRP_{m}_Vehicles",
        pulp.LpMaximize
    )

    # -----------------------------
    # Decision variables
    # -----------------------------
    x = pulp.LpVariable.dicts(
        "x",
        [
            (i, j, k)
            for i in nodes
            for j in nodes
            for k in vehicles
            if i != j
        ],
        cat="Binary"
    )

    y = pulp.LpVariable.dicts(
        "y",
        customers,
        cat="Binary"
    )

    z = pulp.LpVariable.dicts(
        "z",
        vehicles,
        cat="Binary"
    )

    u = pulp.LpVariable.dicts(
        "u",
        [
            (j, k)
            for j in customers
            for k in vehicles
        ],
        lowBound=1,
        upBound=N,
        cat="Continuous"
    )

    # -----------------------------
    # Objective function
    # Maximize completed deliveries, then minimize distance
    # -----------------------------
    model += (
        BIG * pulp.lpSum(y[j] for j in customers)
        -
        pulp.lpSum(
            d[i, j] * x[i, j, k]
            for i in nodes
            for j in nodes
            for k in vehicles
            if i != j
        )
    )

    # -----------------------------
    # Customer visit constraints
    # -----------------------------
    for j in customers:

        model += (
            pulp.lpSum(
                x[i, j, k]
                for i in nodes
                for k in vehicles
                if i != j
            ) == y[j]
        )

        model += (
            pulp.lpSum(
                x[j, i, k]
                for i in nodes
                for k in vehicles
                if i != j
            ) == y[j]
        )

    # -----------------------------
    # Depot start and return
    # -----------------------------
    for k in vehicles:

        model += (
            pulp.lpSum(
                x[depot, j, k]
                for j in customers
            ) == z[k]
        )

        model += (
            pulp.lpSum(
                x[i, depot, k]
                for i in customers
            ) == z[k]
        )

    # -----------------------------
    # Flow conservation
    # -----------------------------
    for k in vehicles:
        for h in customers:

            model += (
                pulp.lpSum(
                    x[i, h, k]
                    for i in nodes
                    if i != h
                )
                ==
                pulp.lpSum(
                    x[h, j, k]
                    for j in nodes
                    if j != h
                )
            )

    # -----------------------------
    # Capacity constraint
    # -----------------------------
    for k in vehicles:

        model += (
            pulp.lpSum(
                q[j] *
                pulp.lpSum(
                    x[i, j, k]
                    for i in nodes
                    if i != j
                )
                for j in customers
            )
            <= Q[k] * z[k]
        )

    # -----------------------------
    # Travel + service time constraint
    # -----------------------------
    for k in vehicles:

        travel_time = pulp.lpSum(
            (d[i, j] / speed) * 60 * x[i, j, k]
            for i in nodes
            for j in nodes
            if i != j
        )

        service_time = pulp.lpSum(
            s[j] *
            pulp.lpSum(
                x[i, j, k]
                for i in nodes
                if i != j
            )
            for j in customers
        )

        model += (
            travel_time + service_time <= H * z[k]
        )

    # -----------------------------
    # MTZ subtour elimination
    # -----------------------------
    for k in vehicles:
        for i in customers:
            for j in customers:
                if i != j:
                    model += (
                        u[i, k] - u[j, k] + N * x[i, j, k]
                        <= N - 1
                    )

    # -----------------------------
    # Symmetry breaking
    # -----------------------------
    for k in range(1, m):
        model += z[k] >= z[k + 1]

    # =================================================
    # SOLVE WITH TIMER
    # =================================================
    start_time = time.time()

    solver = pulp.PULP_CBC_CMD(
        msg=False,
        timeLimit=SOLVER_TIME_LIMIT
    )

    model.solve(solver)

    end_time = time.time()
    runtime_seconds = end_time - start_time

    status = pulp.LpStatus[model.status]

    # =================================================
    # OUTPUT EXTRACTION
    # =================================================
    served = [
        j for j in customers
        if pulp.value(y[j]) is not None and pulp.value(y[j]) > 0.5
    ]

    completed = len(served)
    missed = N - completed
    penalty = missed * PENALTY_PER_MISSED

    vehicle_results = []

    total_distance = 0
    total_elapsed = 0
    total_travel = 0
    total_service = 0

    for k in vehicles:

        if pulp.value(z[k]) is None or pulp.value(z[k]) < 0.5:

            vehicle_results.append({
                "Scenario Vehicles": m,
                "Vehicle": k,
                "Used": "No",
                "Route": [],
                "Completed": 0,
                "Distance miles": 0,
                "Travel min": 0,
                "Service min": 0,
                "Elapsed min": 0,
                "Utilization %": 0
            })

            continue

        route = [depot]
        current = depot
        route_guard = set()

        while True:

            next_node = None

            for j in nodes:
                if current != j:
                    val = pulp.value(x[current, j, k])
                    if val is not None and val > 0.5:
                        next_node = j
                        break

            if next_node is None:
                break

            route.append(next_node)

            if next_node == depot:
                break

            if next_node in route_guard:
                print(f"Warning: loop detected for vehicle {k}")
                break

            route_guard.add(next_node)
            current = next_node

        route_customers = [r for r in route if r != depot]

        route_distance = sum(
            d[a, b] for a, b in zip(route[:-1], route[1:])
        )

        route_travel = route_distance / speed * 60
        route_service = sum(s[j] for j in route_customers)
        route_elapsed = route_travel + route_service
        utilization = route_elapsed / H * 100

        total_distance += route_distance
        total_travel += route_travel
        total_service += route_service
        total_elapsed += route_elapsed

        vehicle_results.append({
            "Scenario Vehicles": m,
            "Vehicle": k,
            "Used": "Yes",
            "Route": route,
            "Completed": len(route_customers),
            "Distance miles": round(route_distance, 2),
            "Travel min": round(route_travel, 2),
            "Service min": round(route_service, 2),
            "Elapsed min": round(route_elapsed, 2),
            "Utilization %": round(utilization, 2)
        })

    summary_result = {
        "Vehicles": m,
        "Status": status,
        "Completed Deliveries": completed,
        "Missed Deliveries": missed,
        "Penalty Cost": round(penalty, 2),
        "Total Fleet Distance miles": round(total_distance, 2),
        "Total Fleet Travel min": round(total_travel, 2),
        "Total Fleet Service min": round(total_service, 2),
        "Total Fleet Elapsed min": round(total_elapsed, 2),
        "Runtime Seconds": round(runtime_seconds, 2),
        "Runtime Minutes": round(runtime_seconds / 60, 2),
        "Runtime Hours": round(runtime_seconds / 3600, 4)
    }

    return summary_result, vehicle_results


# =====================================================
# RUN SCENARIOS: 1, 2, 3, 4, 5 VEHICLES
# =====================================================
summary_all = []
vehicle_all = []

for m in [1, 2, 3, 4, 5]:

    print(f"\nSolving MILP for {m} vehicle(s)...")

    summary, vehicle_result = solve_milp_vrp(m)

    summary_all.append(summary)
    vehicle_all.extend(vehicle_result)

# =====================================================
# CREATE TABLES
# =====================================================
summary_df = pd.DataFrame(summary_all)
vehicle_df = pd.DataFrame(vehicle_all)

print("\n==============================")
print("SUMMARY RESULT")
print("==============================")
print(summary_df)

print("\n==============================")
print("EACH VEHICLE RESULT")
print("==============================")
print(vehicle_df)

# =====================================================
# SAVE OUTPUT
# =====================================================
summary_df.to_csv("MILP_summary_1_to_5_vehicles.csv", index=False)
vehicle_df.to_csv("MILP_each_vehicle_1_to_5_vehicles.csv", index=False)

summary_df.to_excel("MILP_summary_1_to_5_vehicles.xlsx", index=False)
vehicle_df.to_excel("MILP_each_vehicle_1_to_5_vehicles.xlsx", index=False)

print("\nFiles saved:")
print("MILP_summary_1_to_5_vehicles.csv")
print("MILP_each_vehicle_1_to_5_vehicles.csv")
print("MILP_summary_1_to_5_vehicles.xlsx")
print("MILP_each_vehicle_1_to_5_vehicles.xlsx")



Solving MILP for 1 vehicle(s)...

Solving MILP for 2 vehicle(s)...

Solving MILP for 3 vehicle(s)...

Solving MILP for 4 vehicle(s)...

Solving MILP for 5 vehicle(s)...

SUMMARY RESULT
   Vehicles   Status  Completed Deliveries  Missed Deliveries  Penalty Cost  \
0         1  Optimal                    12                 18          21.6   
1         2  Optimal                    21                  9          10.8   
2         3  Optimal                    29                  1           1.2   
3         4  Optimal                    30                  0           0.0   
4         5  Optimal                    30                  0           0.0   

   Total Fleet Distance miles  Total Fleet Travel min  \
0                       34.16                   81.24   
1                       51.16                  121.67   
2                       70.94                  168.71   
3                       81.71                  194.32   
4                       81.71                  194.32 